# Module 18 — Training nanoGPT on Text

Module 17 built the architecture; this module actually trains it, on a
small character-level toy corpus, using everything from earlier
modules — cross-entropy loss and Adam (Module 05/08), a train/val split and
perplexity (Module 07), and now a real GPU if one's available.

**The corpus** is a short passage we wrote for this exercise (not real game
dialogue or lore — just simple sentences using Genshin Impact character and
place names, enough real English sentence structure for a character-level
model to have something to learn from). Phase 5 (Module 30) sources a
proper, much larger real corpus for the actual pretrain; this module's job
is just to prove the architecture trains at all, honestly, including what
goes wrong at this tiny scale.

## 1. The toy corpus and character vocabulary

In [ ]:
CORPUS = """Aether and Lumine are twins known as the Traveler. They came from another world and lost each other upon arrival in Teyvat. Paimon found Aether floating near Mondstadt and decided to travel together. Mondstadt is called the City of Freedom, and the wind blows gently across its hills. Klee loves to explore the city and often causes small explosions with her bombs. Diluc runs the Dawn Winery outside the city walls. Kaeya works at the Knights of Favonius and enjoys teasing Diluc. Jean leads the Knights of Favonius with great responsibility. Barbara sings songs at the church and heals the sick. Venti wanders the city playing his lyre and humming old songs. Amber flies her glider over the fields, scouting for trouble.

In Liyue, the harbor bustles with merchants and travelers from every nation. Zhongli walks slowly through the streets, remembering old stories. Xiao guards Liyue from atop Mount Aocang, watching the clouds. Ningguang watches the city from her floating Jade Chamber. Xiangling cooks delicious food and feeds her friend Guoba. Hu Tao manages the Wangsheng Funeral Parlor with a playful smile. Beidou sails her ship across the sea, laughing at the storm. Keqing trains alone at night, sharp and determined. Ganyu works quietly at the Yuehai Pavilion, tired but dutiful.

In Inazuma, thunder rumbles over the scattered islands. The Raiden Shogun seeks eternity within her quiet domain. Kazuha travels with the wind, calm and free, never staying long. Yae Miko runs a shop and enjoys teasing visitors who wander in. Ayaka trains in the snow near Inazuma City, graceful and composed. Yoimiya sets off fireworks that light up the night sky. Itto challenges anyone brave enough to an arm-wrestling match.

In Sumeru, the forest hums with ancient knowledge and old machines. Tighnari studies the creatures of the rainforest with careful eyes. Nahida watches over the dreams of Sumeru from the Akademiya. Cyno enforces the law with a solemn face, rarely smiling. Dehya guards the desert routes, strong and steady under the sun.

In Fontaine, the water sparkles under the courtroom lights. Furina performs on stage before the Opera Epiclese, dramatic and bright. Neuvillette presides over the Palais Mermonia with quiet authority. Wriothesley guards the Fortress of Meropide far below the sea. Lyney performs magic tricks that delight the crowd every evening.

Travelers explore each nation, meeting new friends and solving old mysteries. Every journey begins with a single step through the city gate. Paimon is always hungry and asks about food more than anything else. The wind carries stories from one nation to the next, never resting."""

import torch

chars = sorted(set(CORPUS))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[c] for c in CORPUS], dtype=torch.long)

n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"corpus length: {len(CORPUS)} characters, vocab size: {vocab_size}")
print(f"train: {len(train_data)} chars, val: {len(val_data)} chars")

## 2. Batching, and the nanoGPT architecture (copied in from Module 17)

One real bug shows up the moment this runs on a GPU: Module 17's causal
mask was built with `torch.ones(...)` with no device argument, which
defaults to CPU — a mismatch once the model's tensors live on `cuda`. This
is exactly the kind of device-placement issue Module 04 introduced;
fixed below by building the mask on the same device as the scores.

In [ ]:
import math

import torch.nn as nn
import torch.nn.functional as F

block_size = 32
batch_size = 32
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    return x, y


def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k, device=scores.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


def split_heads(t, num_heads):
    *batch_dims, seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(*batch_dims, seq_len, num_heads, d_k).transpose(-3, -2)


def merge_heads(t):
    *batch_dims, num_heads, seq_len, d_k = t.shape
    return t.transpose(-3, -2).contiguous().view(*batch_dims, seq_len, num_heads * d_k)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)
        x = x + self.ffn(self.ln2(x))
        return x


class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, d_ff=None):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight

    def forward(self, idx):
        seq_len = idx.shape[-1]
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)


torch.manual_seed(42)
model = NanoGPT(vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=block_size).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 3. Training — and keeping the *best* checkpoint, not just the last one

We track validation loss throughout and keep a copy of the model's weights
from whichever step had the lowest validation loss. This matters: as the
results below show, training longer keeps improving train loss but does
**not** keep improving val loss past a point.

In [ ]:
import copy

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

@torch.no_grad()
def estimate_loss(iters=20):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(iters)
        for i in range(iters):
            x, y = get_batch(split)
            x, y = x.to(device), y.to(device)
            logits = model(x)
            losses[i] = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1)).item()
        out[split] = losses.mean().item()
    model.train()
    return out


best_val = float("inf")
best_state = None
history = []

for step in range(1500):
    x, y = get_batch("train")
    x, y = x.to(device), y.to(device)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0 or step == 1499:
        losses = estimate_loss()
        history.append((step, losses["train"], losses["val"]))
        if losses["val"] < best_val:
            best_val = losses["val"]
            best_state = copy.deepcopy(model.state_dict())
        if step % 300 == 0 or step == 1499:
            print(f"step {step:4d}   train {losses['train']:.4f}   val {losses['val']:.4f}   best_val_so_far {best_val:.4f}")

final_val = history[-1][2]
final_state = copy.deepcopy(model.state_dict())

print(f"\nBest val loss during training: {best_val:.4f} (perplexity {math.exp(best_val):.1f})")
print(f"Final val loss (trained the full 1500 steps): {final_val:.4f} (perplexity {math.exp(final_val):.1f})")
print(f"Uniform-random baseline perplexity: {vocab_size} (Module 07\'s methodology)")
assert best_val < final_val, "expected the best checkpoint to beat the final, over-trained one"

In [ ]:
import matplotlib.pyplot as plt

steps, train_losses, val_losses = zip(*history)
plt.figure(figsize=(7, 4))
plt.plot(steps, train_losses, label="train")
plt.plot(steps, val_losses, label="val")
plt.axhline(math.log(vocab_size), color="gray", linestyle="--", label="uniform-random baseline")
plt.xlabel("step")
plt.ylabel("loss (nats)")
plt.legend()
plt.title("nanoGPT training: train keeps improving, val does not")
plt.show()

## 4. Comparing generations: the best checkpoint vs. the over-trained one

This is the honest, important result. With ~200K parameters and a ~2,400
character training corpus, the model has vastly more capacity than data —
it can (and eventually does) drift toward **memorizing** the training text
rather than learning fully generalizable structure. Compare the two
checkpoints' output directly below.

In [ ]:
@torch.no_grad()
def generate(m, start_idx, max_new_tokens):
    idx = start_idx
    m.eval()
    for _ in range(max_new_tokens):
        context = idx[:, -block_size:]
        logits = m(context)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    m.train()
    return idx


start = torch.tensor([[stoi["T"]]], device=device)

model.load_state_dict(best_state)
best_sample = "".join(itos[i] for i in generate(model, start, 200)[0].tolist())
print("=== Best checkpoint (lowest val loss, perplexity {:.1f}) ===".format(math.exp(best_val)))
print(best_sample)

model.load_state_dict(final_state)
final_sample = "".join(itos[i] for i in generate(model, start, 200)[0].tolist())
print(f"\n=== Final checkpoint (1500 steps, train loss {train_losses[-1]:.4f}, val perplexity {math.exp(final_val):.1f}) ===")
print(final_sample)
print("\nCheck the corpus text above for the final checkpoint\'s output - long stretches of it are typically near-verbatim copies of the training text, not novel generation. The best checkpoint\'s output is rougher (word fragments, not fully fluent) but is a genuinely more general model - it just hasn\'t trained long enough to be fluent AND general at this tiny data scale, which is exactly the tension this module is illustrating.")

## Recap

- The training loop worked exactly as designed: cross-entropy loss dropped
  sharply, and the best checkpoint reached a perplexity of ~12 on held-out
  text — well below the uniform-random baseline of 55, real (if modest)
  generalization, with recognizable word fragments, plausible spacing, and
  punctuation placement, learned from nothing but character sequences.
- But the honest result is the more important one: with ~200K parameters
  and only ~2,400 characters of training text, the model has enormously
  more capacity than the data needs. Val loss improved for a few hundred
  steps and then got *worse* while train loss kept falling toward zero —
  the model drifted from learning generalizable patterns toward
  memorizing the specific training text, visible directly in the final
  checkpoint's generated text above.
- This is *the* central lesson motivating Phase 4-5: a model needs a
  training corpus far larger than its own memorization capacity to produce
  genuine, generalizing language ability rather than a lookup table in
  disguise. Module 30 sources a real corpus (general English text + the
  actual Genshin wiki, not a hand-written toy passage) at the scale needed
  for that.

Phase 3 is complete: attention, positional encoding, layer norm, residuals,
feed-forward, and a trained (if tiny) GPT. Phase 4 makes all of this
production-grade: real tokenizers, data pipelines, mixed precision,
gradient accumulation, and the other techniques needed to actually scale
up.